In [2]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

## Impliment Rag using own text data

### step 1: create data

In [4]:
from langchain_core.documents import Document

data = """
Text data (or textual data) refers to information expressed in written language form, 
including emails, social media posts, documents, and transcripts.  
It is characterized as unstructured or semi-structured because it does not fit neatly into rows and columns like traditional structured data. 

To extract meaningful insights, this data requires processing through Natural Language Processing (NLP) and text analytics.
Key techniques include sentiment analysis, named entity recognition, 
and text classification, which allow computers to interpret human language patterns.  Common examples of text data
include books, news articles, customer reviews, and legal documents.
"""

doc = [Document(page_content=data,metadata={"source":"google"})]
doc

[Document(metadata={'source': 'google'}, page_content='\nText data (or textual data) refers to information expressed in written language form, \nincluding emails, social media posts, documents, and transcripts.  \nIt is characterized as unstructured or semi-structured because it does not fit neatly into rows and columns like traditional structured data. \n\nTo extract meaningful insights, this data requires processing through Natural Language Processing (NLP) and text analytics.\nKey techniques include sentiment analysis, named entity recognition, \nand text classification, which allow computers to interpret human language patterns.  Common examples of text data\ninclude books, news articles, customer reviews, and legal documents.\n')]

### Step 2: splitting the documents into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

chunk = splitter.split_documents(doc)

chunk

[Document(metadata={'source': 'google'}, page_content='Text data (or textual data) refers to information expressed in written language form, \nincluding emails, social media posts, documents, and transcripts.  \nIt is characterized as unstructured or semi-structured because it does not fit neatly into rows and columns like traditional structured data.'),
 Document(metadata={'source': 'google'}, page_content='To extract meaningful insights, this data requires processing through Natural Language Processing (NLP) and text analytics.\nKey techniques include sentiment analysis, named entity recognition, \nand text classification, which allow computers to interpret human language patterns.  Common examples of text data\ninclude books, news articles, customer reviews, and legal documents.')]

### Step 3: Create Embedding

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings_model = HuggingFaceEmbeddings(model_name=model_name)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1107.43it/s]


### Step 4: Store

In [20]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=chunk,embedding=embeddings_model)

### Step 5: Semantic Search

In [22]:
context = vectorstore.similarity_search("What is text data?",k=2)

In [23]:
res = model.invoke(f"Answer the question based on the context below:\n\nContext: {context}\n\nQuestion: What is text data?")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [25]:
res.text

'Based on the provided context, text data (or textual data) refers to information expressed in written language form. It is characterized as unstructured or semi-structured because it does not fit neatly into traditional rows and columns like structured data. \n\nExamples of text data include emails, social media posts, documents, transcripts, books, news articles, customer reviews, and legal documents.'